# 🦥 Fine-tune & Evaluate Gemma 4 Vision on Google Colab

This notebook provides a complete pipeline for fine-tuning and evaluating the Gemma 4 Vision (E2B) model using Unsloth.

## 1. Install Dependencies
We use Unsloth for 2x faster training and significantly lower VRAM usage.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## 2. Mount Google Drive
Mount your drive to access the dataset and save the trained model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration & Helpers

In [ ]:
import os
import json
import torch
import time
from PIL import Image
from datasets import Dataset
from transformers import TextStreamer
from unsloth import FastVisionModel, get_chat_template
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig, SFTTrainer

# --- CONFIGURATION ---
LOCAL_BASE_PATH = "/home/quangnhvn34/dev/me/InsureVN/data/health_insurance/health_insurance_extracted"
DATA_DIR = "/content/drive/MyDrive/02_AI_Research_&_Data/health_insurance_extracted"

# Save all outputs in the same Google Drive folder as the data
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs_gemma4_finetuned")
SAVE_DIR = os.path.join(DATA_DIR, "gemma4-e2b-finetuned-lora")
GGUF_DIR = os.path.join(DATA_DIR, "gemma4-gguf")

GGUF_METHOD = "q4_k_m"
NUM_EPOCHS = 2

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def fix_path(old_path):
    """Converts local machine paths to Colab Google Drive paths."""
    if isinstance(old_path, str) and old_path.startswith(LOCAL_BASE_PATH):
        return old_path.replace(LOCAL_BASE_PATH, DATA_DIR)
    # Handle case where it might be relative or slightly different
    basename = os.path.basename(str(old_path))
    return os.path.join(DATA_DIR, basename)

def load_jsonl_as_unsloth_dataset(jsonl_path):
    converted = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            formatted_messages = []
            for msg in data["messages"]:
                if msg["role"] == "user":
                    new_content = []
                    for item in msg["content"]:
                        if item["type"] == "image_path":
                            corrected_image_path = fix_path(item["content"])
                            new_content.append({"type": "image", "image": corrected_image_path})
                        elif item["type"] == "text":
                            new_content.append({"type": "text", "text": item["content"]})
                    formatted_messages.append({"role": "user", "content": new_content})
                elif msg["role"] == "assistant":
                    formatted_messages.append({"role": "assistant", "content": [{"type": "text", "text": msg["content"]}]})
            converted.append({"messages": formatted_messages})
    return converted

def format_user_messages_inference(raw_messages):
    """Convert JSONL user message format to Unsloth inference format."""
    messages = []
    for msg in raw_messages:
        if msg["role"] != "user":
            continue
        new_content = []
        for item in msg["content"]:
            if item["type"] == "image_path":
                new_content.append({"type": "image", "image": fix_path(item["content"])})
            elif item["type"] == "text":
                new_content.append({"type": "text", "text": item["content"]})
        messages.append({"role": "user", "content": new_content})
    return messages

## 4. Load Model & Processor

In [ ]:
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    target_modules="all-linear",
)

processor = get_chat_template(processor, "gemma-4")

## 5. Training

In [ ]:
jsonl_file = os.path.join(DATA_DIR, "oumi_vlm_train.jsonl")
converted_dataset = load_jsonl_as_unsloth_dataset(jsonl_file)
print(f"Loaded {len(converted_dataset)} training samples.")

trainer = SFTTrainer(
    model=model,
    train_dataset=converted_dataset,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)

trainer.train()

## 6. Save & Export

In [ ]:
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"Adapters saved to {SAVE_DIR}")

if GGUF_DIR:
    print(f"Exporting to GGUF ({GGUF_METHOD})...")
    model.save_pretrained_gguf(
        GGUF_DIR,
        processor.tokenizer,
        quantization_method=GGUF_METHOD,
    )
    print(f"GGUF model saved to {GGUF_DIR}")

## 7. Evaluation
Run inference on test samples to verify performance.

In [ ]:
# Switch to inference mode
FastVisionModel.for_inference(model)

test_file = os.path.join(DATA_DIR, "oumi_vlm_test.jsonl")
test_examples = []
with open(test_file, "r", encoding="utf-8") as f:
    for line in f:
        test_examples.append(json.loads(line))

print(f"Loaded {len(test_examples)} test samples.")

def run_eval(num_samples=3):
    for i in range(min(num_samples, len(test_examples))):
        sample = test_examples[i]
        messages = format_user_messages_inference(sample["messages"])
        
        # Extract image path
        image_path = next((item["image"] for item in messages[0]["content"] if item["type"] == "image"), None)
        if not image_path or not os.path.exists(image_path):
            print(f"Image not found: {image_path}")
            continue
            
        print(f"\n--- Sample {i+1} ---")
        image = Image.open(image_path)
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
        
        print("Assistant Response:")
        text_streamer = TextStreamer(processor, skip_prompt=True)
        _ = model.generate(
            **inputs, 
            streamer=text_streamer, 
            max_new_tokens=1024, 
            use_cache=True, 
            temperature=1.0, 
            top_p=0.95, 
            top_k=64
        )
        
        # Show Ground Truth
        ground_truth = next((msg["content"] for msg in sample["messages"] if msg["role"] == "assistant"), "None")
        print(f"\nGround Truth:\n{ground_truth}")
        print("-" * 40)



In [ ]:
run_eval(3)